## Generate Truth Dataset for validation - Class A Set 2

In [27]:
import os
import fitz  # PyMuPDF
from ocr_tamil.ocr import OCR

# Suppress warnings to keep output clean
import warnings
warnings.filterwarnings('ignore')

# Fix for timm Attention layer compatibility issue
import torch
import torch.nn as nn
from timm.layers.attention import Attention

# Monkey patch the Attention.forward method to handle missing norm
original_forward = Attention.forward

def patched_forward(self, x, attn_mask=None):
    B, N, C = x.shape
    qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
    q, k, v = qkv[0], qkv[1], qkv[2]
    attn = (q @ k.transpose(-2, -1)) * self.scale
    if attn_mask is not None:
        attn = attn.view(B * self.num_heads, N, N) + attn_mask
        attn = attn.view(B, self.num_heads, N, N)
    attn = attn.softmax(dim=-1)
    attn = self.attn_drop(attn)
    x = (attn @ v).transpose(1, 2).reshape(B, N, C)
    
    # Skip norm if it doesn't exist (compatibility fix)
    if hasattr(self, 'norm'):
        x = self.norm(x)
    
    x = self.proj(x)
    x = self.proj_drop(x)
    return x

Attention.forward = patched_forward
print("✓ Applied timm Attention compatibility patch")

✓ Applied timm Attention compatibility patch


### Initialize OCR model

In [28]:
ocr_detect_tamil = OCR(
    detect=True,
    lang=["tamil", "english"],
    batch_size=128,
    tamil_model_path=r"/Users/vidhya/Downloads/tamil_ocr-main/ocr_tamil/model_weights/parseq_tamil_v3.pt",
    eng_model_path=r"/Users/vidhya/Downloads/tamil_ocr-main/ocr_tamil/model_weights/craft_mlt_25k.pth"
)

In [31]:
pdf_path = "../data/FIR/SZ/ATC, Cr. No. 04 of 2017.pdf"

# Open PDF
pdf_document = fitz.open(pdf_path)

all_texts = []
for page_num in range(len(pdf_document)):
    page = pdf_document.load_page(page_num)
    print(f"Rendering page {page_num + 1}",page)
    pix = page.get_pixmap()
    print(f"Loaded page {page_num + 1}",pix)
    print(f"Processing page {page_num + 1}/{len(pdf_document)}")
    image_path = f"../data/temp/temp_page_{page_num}.png"
    print(f"Saving temporary image to {image_path}")
    pix.save(image_path)

    try:
        # OCR the temp image
        texts = ocr_detect_tamil.predict(image_path)
        full_text = " ".join(texts[0])

        # Format text: 6 words per line
        words = full_text.split()
        formatted_text = "\n".join(
            [" ".join(words[i:i+6]) for i in range(0, len(words), 6)]
        )
        all_texts.append(formatted_text)

    finally:
        # Delete temp file safely
        if os.path.exists(image_path):
            os.remove(image_path)

# Combine all pages
final_text = "\n\n".join(all_texts)
print(final_text)

# Save extracted text to file
output_file_path = "../data/ground_truth/SZ/ATC, Cr. No. 04 of 2017.txt"
with open(output_file_path, "w", encoding="utf-8") as f:
    f.write(final_text)

print(f"Final text saved to {output_file_path}")


Rendering page 1 page 0 of ../data/FIR/SZ/ATC, Cr. No. 04 of 2017.pdf
Loaded page 1 Pixmap(DeviceRGB, (0, 0, 595, 842), 0)
Processing page 1/3
Saving temporary image to ../data/temp/temp_page_0.png
Rendering page 2 page 1 of ../data/FIR/SZ/ATC, Cr. No. 04 of 2017.pdf
Loaded page 2 Pixmap(DeviceRGB, (0, 0, 595, 842), 0)
Processing page 2/3
Saving temporary image to ../data/temp/temp_page_1.png
Rendering page 2 page 1 of ../data/FIR/SZ/ATC, Cr. No. 04 of 2017.pdf
Loaded page 2 Pixmap(DeviceRGB, (0, 0, 595, 842), 0)
Processing page 2/3
Saving temporary image to ../data/temp/temp_page_1.png
Rendering page 3 page 2 of ../data/FIR/SZ/ATC, Cr. No. 04 of 2017.pdf
Loaded page 3 Pixmap(DeviceRGB, (0, 0, 595, 842), 0)
Processing page 3/3
Saving temporary image to ../data/temp/temp_page_2.png
Rendering page 3 page 2 of ../data/FIR/SZ/ATC, Cr. No. 04 of 2017.pdf
Loaded page 3 Pixmap(DeviceRGB, (0, 0, 595, 842), 0)
Processing page 3/3
Saving temporary image to ../data/temp/temp_page_2.png
FIRST INFO

In [32]:
import os
import fitz
from pathlib import Path

BASE_INPUT_DIR = "../data/FIR"
BASE_OUTPUT_DIR = "../data/ground_truth"
TEMP_DIR = "../data/temp"

os.makedirs(TEMP_DIR, exist_ok=True)

def process_pdf(pdf_path, output_txt_path):
    pdf_document = fitz.open(pdf_path)
    all_texts = []

    for page_num in range(len(pdf_document)):
        page = pdf_document.load_page(page_num)
        pix = page.get_pixmap()

        temp_image_path = os.path.join(TEMP_DIR, f"temp_page_{page_num}.png")
        pix.save(temp_image_path)

        try:
            texts = ocr_detect_tamil.predict(temp_image_path)
            full_text = " ".join(texts[0])

            # Format as 6 words per line
            words = full_text.split()
            formatted_text = "\n".join(
                [" ".join(words[i:i+6]) for i in range(0, len(words), 6)]
            )

            all_texts.append(formatted_text)

        finally:
            if os.path.exists(temp_image_path):
                os.remove(temp_image_path)

    final_text = "\n\n".join(all_texts)

    # Write output TXT
    with open(output_txt_path, "w", encoding="utf-8") as f:
        f.write(final_text)

    print(f"✔ Saved: {output_txt_path}")


def ensure_output_path(input_pdf_path):
    """
    Given an input like  FIR/SZ/sub/X.pdf
    Create ground_truth/SZ/sub/X.txt
    """
    relative_path = os.path.relpath(input_pdf_path, BASE_INPUT_DIR)
    txt_relative = Path(relative_path).with_suffix(".txt")

    output_path = Path(BASE_OUTPUT_DIR) / txt_relative
    output_path.parent.mkdir(parents=True, exist_ok=True)

    return str(output_path)


# --------------------------------------------------------
# 🚀 MAIN LOOP — Iterate through all PDFs recursively
# --------------------------------------------------------
for root, dirs, files in os.walk(BASE_INPUT_DIR):
    for file in files:
        if file.lower().endswith(".pdf"):
            pdf_path = os.path.join(root, file)
            output_txt_path = ensure_output_path(pdf_path)

            print(f"\nProcessing: {pdf_path}")
            process_pdf(pdf_path, output_txt_path)

print("\n🎉 Completed processing all FIR PDFs.")



Processing: ../data/FIR/NZ/CBCID,Vellore 2022 Accidental case.pdf
✔ Saved: ../data/ground_truth/NZ/CBCID,Vellore 2022 Accidental case.txt

Processing: ../data/FIR/NZ/Metro 2022 Land Cheating case.pdf
✔ Saved: ../data/ground_truth/NZ/CBCID,Vellore 2022 Accidental case.txt

Processing: ../data/FIR/NZ/Metro 2022 Land Cheating case.pdf
✔ Saved: ../data/ground_truth/NZ/Metro 2022 Land Cheating case.txt

Processing: ../data/FIR/NZ/Vellore 2020 Accidental case.pdf
✔ Saved: ../data/ground_truth/NZ/Metro 2022 Land Cheating case.txt

Processing: ../data/FIR/NZ/Vellore 2020 Accidental case.pdf
✔ Saved: ../data/ground_truth/NZ/Vellore 2020 Accidental case.txt

Processing: ../data/FIR/NZ/Hanging case.pdf
✔ Saved: ../data/ground_truth/NZ/Vellore 2020 Accidental case.txt

Processing: ../data/FIR/NZ/Hanging case.pdf
✔ Saved: ../data/ground_truth/NZ/Hanging case.txt

Processing: ../data/FIR/NZ/Metro 2018 Murder case.pdf
✔ Saved: ../data/ground_truth/NZ/Hanging case.txt

Processing: ../data/FIR/NZ/Metr